# Component 2: SpatialSoftmax Pooling

After ResNet18 extracts feature maps, we need to convert them into a compact vector. The standard approach is **Global Average Pooling (GAP)**, which simply averages each channel. But GAP throws away all spatial information -- it tells you *how much* of a feature is present, but not *where*.

**SpatialSoftmax** is a smarter pooling strategy. For each feature channel, it:
1. Applies softmax over the spatial dimensions (H x W) to get an attention map
2. Uses this attention map as weights to compute the expected (x, y) coordinate

The result: each channel produces a single (x, y) keypoint, giving us a set of **learned spatial keypoints** that track semantically meaningful locations in the image.

**In Diffusion Policy:** ResNet18 outputs 512 channels of 3×3 feature maps. A 1×1 conv reduces this to `num_kp` channels (default 32), then SpatialSoftmax extracts 32 keypoints → **64 values** (32 × 2). You *could* skip the reduction and get 512 × 2 = 1,024 values, but 32 keypoints suffices and keeps the conditioning vector compact.

**The Math:**
- Given feature map $F_k$ of shape $(H, W)$ for channel $k$:
  - $\alpha_k(i, j) = \frac{\exp(F_k(i,j))}{\sum_{i',j'} \exp(F_k(i',j'))}$ (softmax over spatial dims)
  - $x_k = \sum_{i,j} \alpha_k(i,j) \cdot j_{\text{coord}}$ (expected x-coordinate)
  - $y_k = \sum_{i,j} \alpha_k(i,j) \cdot i_{\text{coord}}$ (expected y-coordinate)

In [ ]:
!pip install -q torch torchvision matplotlib numpy 2>&1 | tail -3

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import torchvision.models as models
import torch.nn as nn

# Load pretrained ResNet18 (same backbone used in diffusion policy)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device).eval()

# Extract just the backbone layers (remove avgpool + fc, same as diffusion policy)
backbone = nn.Sequential(
    resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
    resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
).to(device).eval()

# Create a synthetic test image (96x96, simulating a robot workspace)
torch.manual_seed(42)
test_img = torch.zeros(1, 3, 96, 96)
# Red object (top-left)
test_img[0, 0, 15:45, 15:45] = 0.9
test_img[0, 1, 15:45, 15:45] = 0.2
test_img[0, 2, 15:45, 15:45] = 0.15
# Blue target zone (bottom-right)
test_img[0, 0, 55:80, 55:80] = 0.15
test_img[0, 1, 55:80, 55:80] = 0.3
test_img[0, 2, 55:80, 55:80] = 0.9
# Green table background
test_img[0, 1] += 0.15
test_img += torch.randn_like(test_img) * 0.05
test_img = test_img.clamp(0, 1)

# ImageNet normalization (same as diffusion policy)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
test_img_norm = ((test_img - MEAN) / STD).to(device)

# Get backbone features
with torch.no_grad():
    backbone_features = backbone(test_img_norm)
    
B, C, H, W = backbone_features.shape
print(f"Input image:       {test_img.shape} (B, C, H, W)")
print(f"Backbone output:   {backbone_features.shape}")
print(f"  → {C} channels × {H}×{W} spatial = {C*H*W} values")

## Implementing SpatialSoftmax from Scratch

Let us build the SpatialSoftmax operation step by step so we understand exactly what it does, then compare with the policy's built-in version.

In [ ]:
# Step 1: Define number of keypoints
# In diffusion policy, a 1x1 conv reduces 512 channels → num_kp channels
num_keypoints = 32
print(f"Backbone channels: {C}")
print(f"Num keypoints:     {num_keypoints}")
print(f"Output dimension:  {num_keypoints * 2} (x,y per keypoint)\n")

# Step 2: Create the 1x1 conv (channel reduction) — same as diffusion policy
reduce_conv = nn.Conv2d(C, num_keypoints, kernel_size=1).to(device)
nn.init.xavier_uniform_(reduce_conv.weight)

with torch.no_grad():
    reduced_features = reduce_conv(backbone_features)  # (B, num_kp, H, W)
print(f"After 1×1 conv: {reduced_features.shape}")

# Step 3: Create coordinate grids normalized to [-1, 1]
y_coords = torch.linspace(-1, 1, H, device=device)
x_coords = torch.linspace(-1, 1, W, device=device)
yy, xx = torch.meshgrid(y_coords, x_coords, indexing="ij")
print(f"Coordinate grid: {xx.shape} with range [{x_coords[0]:.1f}, {x_coords[-1]:.1f}]")

# Step 4: Apply softmax over spatial dimensions
flat = reduced_features.reshape(B, num_keypoints, -1)      # (B, K, H*W)
attention_weights = torch.softmax(flat, dim=-1)              # (B, K, H*W)
attention_maps = attention_weights.reshape(B, num_keypoints, H, W)

print(f"\nAttention maps shape: {attention_maps.shape}")
print(f"Sum per channel (should be 1.0): {attention_weights[0, 0].sum().item():.6f}")

# Step 5: Compute expected (x, y) coordinates
expected_x = (attention_maps * xx.unsqueeze(0).unsqueeze(0)).sum(dim=(-2, -1))  # (B, K)
expected_y = (attention_maps * yy.unsqueeze(0).unsqueeze(0)).sum(dim=(-2, -1))  # (B, K)

# Step 6: Concatenate into final output
keypoints = torch.cat([expected_x, expected_y], dim=-1)  # (B, K*2)
print(f"\n--- Result ---")
print(f"Keypoints shape: {keypoints.shape}  →  {num_keypoints} keypoints × 2 coords = {num_keypoints*2} values")
print(f"First 6 x-coords: {expected_x[0, :6].detach().cpu().numpy().round(3)}")
print(f"First 6 y-coords: {expected_y[0, :6].detach().cpu().numpy().round(3)}")
print(f"\nEach (x,y) pair tells us WHERE a learned feature is located in the image!")

In [ ]:
# Compare: What if we used all 512 channels without reduction?
print("=== With 1×1 conv reduction (what diffusion policy does) ===")
print(f"  512 channels → {num_keypoints} keypoints → {num_keypoints*2} output values")
print()
print("=== Without reduction (all 512 channels) ===")
print(f"  512 channels → 512 keypoints → {512*2} output values")
print()
print("The 1×1 conv acts as a learned bottleneck:")
print("  - Forces the network to pick the most useful spatial features")
print("  - Keeps the conditioning vector compact (64 vs 1024 values)")
print("  - 32 keypoints is enough to represent object positions, gripper, obstacles")

# Let's also try it with all 512 channels for comparison
with torch.no_grad():
    flat_all = backbone_features.reshape(B, C, -1)
    attn_all = torch.softmax(flat_all, dim=-1).reshape(B, C, H, W)
    ex_all = (attn_all * xx.unsqueeze(0).unsqueeze(0)).sum(dim=(-2, -1))
    ey_all = (attn_all * yy.unsqueeze(0).unsqueeze(0)).sum(dim=(-2, -1))
    kp_all = torch.cat([ex_all, ey_all], dim=-1)

print(f"\nAll-512 output shape: {kp_all.shape} ({kp_all.shape[1]} values)")
print(f"Reduced-32 output shape: {keypoints.shape} ({keypoints.shape[1]} values)")

In [ ]:
# Visualize keypoints overlaid on the original image
kp_np = keypoints[0].detach().cpu().numpy()
n_kp = len(kp_np) // 2
kp_x = kp_np[:n_kp]   # x-coordinates in [-1, 1]
kp_y = kp_np[n_kp:]   # y-coordinates in [-1, 1]

# Convert from [-1, 1] to pixel coordinates
pixel_x = (kp_x + 1) / 2 * 96
pixel_y = (kp_y + 1) / 2 * 96

img_display = test_img[0].permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Keypoints on image
axes[0].imshow(img_display)
axes[0].scatter(pixel_x, pixel_y, c=np.arange(n_kp), cmap="rainbow", s=80,
                edgecolors="white", linewidths=1.5, zorder=5)
for i in range(0, n_kp, 4):  # label every 4th
    axes[0].annotate(str(i), (pixel_x[i], pixel_y[i]), fontsize=7,
                     ha="center", va="bottom", color="white", fontweight="bold")
axes[0].set_title(f"{n_kp} SpatialSoftmax Keypoints", fontweight="bold")
axes[0].axis("off")

# Plot 2: GAP output (no spatial info)
with torch.no_grad():
    gap_output = backbone_features.mean(dim=(-2, -1))  # (B, 512)
axes[1].bar(range(32), gap_output[0, :32].cpu().numpy(), color="#5B8CB8", alpha=0.7)
axes[1].set_title("Global Avg Pooling (first 32 ch)\nMagnitudes only — NO position", fontweight="bold")
axes[1].set_xlabel("Channel")
axes[1].set_ylabel("Average activation")

# Plot 3: SpatialSoftmax output
axes[2].bar(range(len(kp_np)), kp_np, color="#D97757", alpha=0.7)
axes[2].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[2].set_title(f"SpatialSoftmax ({len(kp_np)}d)\n(x,y) coords = WHERE features are", fontweight="bold")
axes[2].set_xlabel("Dimension (first 32=x, last 32=y)")
axes[2].set_ylabel("Coordinate [-1, 1]")

plt.tight_layout()
plt.show()

print(f"GAP: {gap_output.shape[1]}d → tells HOW MUCH of each feature, but not WHERE")
print(f"SSM: {len(kp_np)}d → tells exactly WHERE each feature is located")

In [ ]:
# Visualize the attention heatmaps for individual keypoint channels
# These show WHERE in the image each keypoint is "looking"

with torch.no_grad():
    K = reduced_features.shape[1]
    flat_r = reduced_features.reshape(1, K, -1)
    attn = torch.softmax(flat_r, dim=-1).reshape(1, K, H, W)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("SpatialSoftmax Attention Heatmaps\n(brighter = higher weight for keypoint location)",
             fontsize=14, fontweight="bold")

for idx, ax in enumerate(axes.flat):
    ch_idx = idx * (K // 8)
    if ch_idx >= K:
        ax.axis("off")
        continue
    
    heatmap = attn[0, ch_idx].cpu().numpy()
    
    # Show original image with heatmap overlay
    ax.imshow(img_display)
    # Upsample heatmap to image size for overlay
    heatmap_up = np.kron(heatmap, np.ones((96 // H, 96 // W)))[:96, :96]
    ax.imshow(heatmap_up, cmap="hot", alpha=0.6)
    
    # Mark the keypoint
    if ch_idx < n_kp:
        ax.scatter([pixel_x[ch_idx]], [pixel_y[ch_idx]], c="cyan", s=100,
                   marker="+", linewidths=2)
    
    ax.set_title(f"Keypoint {ch_idx}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Each heatmap shows the attention distribution for one keypoint.")
print("The cyan '+' marks the expected (x,y) coordinate — the weighted")
print("average of all positions according to the attention map.")

## TODO Exercise

**Change `num_keypoints` from 32 to 8. What happens to the feature quality?**

1. Modify the number of keypoints in the SpatialSoftmax (you will need to create a new 1x1 conv layer):

```python
# TODO: Create a new SpatialSoftmax with only 8 keypoints
# new_conv = torch.nn.Conv2d(512, 8, kernel_size=1).to(device)
# reduced_8 = new_conv(backbone_features)
# ... apply softmax and compute expected coordinates ...
# Plot the 8 keypoints on the image -- do they still cover the important parts?
```

2. With only 8 keypoints (16 values), can the model still represent the scene adequately? Think about what information might be lost.

3. Try with 64 keypoints (128 values). Is there a point of diminishing returns?